# Lab 6: Performance Metrics and Validation

**Part of the Iceland ML Course: Sentinel-2 Classification Project**

This lecture is part of the course [Machine Learning for Earth Observation powered by Supercomputers (REI506M, 2023)](https://ugla.hi.is/kennsluskra/index.php?tab=nam&chapter=namskeid&id=71151920236&kennsluar=2023)

---

## Project Milestone Overview

| Lab | Milestone | Status |
|-----|-----------|--------|
| Lab 4.1 | Training on Sentinel-2 Data | ✅ Previous |
| Lab 5 | Distributed Training (Multi-GPU) | ✅ Previous |
| Lab 6 | **Validation & Performance Metrics** | 🔄 **Current** |
| Lab 7 | Foundation Models & TerraToRCH | ⬜ Next |

---

## Project Context

**Inputs (From Lab 5):**
- Trained transformer model checkpoint
- Validation dataset (features + CORINE labels)
- Training metrics and logs

**What You'll Do:**
- Load your trained model
- Run inference on validation set
- Calculate accuracy metrics (OA, PA, UA, confusion matrix)
- Compare with established land cover products
- Visualize results in QGIS

**Output:**
- Validation report with metrics
- Comparison with WorldCover and Esri products
- Quantitative assessment of model performance

---

**Outline**

[TOC]

## Lecture Content
In this lab, we explored the vital process of validating land cover maps derived from satellite data and machine learning. Covering qualitative and quantitative validation methods, we emphasized the significance of reliable ground truth data.

## Learning Objectives

- Be able to compare the LC maps with other datasets.
- Be able to assess the accuracy of a classification through the confusion matrix
- Understand what the terms overall accuracy/error, producer's and user's accuracy, and error of omission and commission, are referring to, and be able to calculate them for different classification classes.

---

## Other Land Cover Maps

### WorldCover by ESA

[WorldCover2021](https://worldcover2021.esa.int/), developed by the European Space Agency (ESA), stands as a global land cover map providing comprehensive coverage of Earth's surface. Leveraging state-of-the-art satellite data, WorldCover offers a detailed and up-to-date portrayal of land cover types worldwide. Its significance lies in serving as a benchmark for land cover mapping initiatives, enabling comparisons and validations on a global scale. 

- fast generation and validation of a world land cover based on Sentinel-2 and Sentinel-1 constellations
- 11 land cover classes, description is [here](https://worldcover2021.esa.int/documentation) 
- 10m resolution
- 77% overall accuracy, independently validated
- No need to download, find the name [here](https://viewer.esa-worldcover.org/worldcover/), mark it and copy it from  /p/project/training2328/tian1/  

![](https://gitlab.jsc.fz-juelich.de/hedgedoc/uploads/f07043d1-9fdf-47af-b4df-d95587d4677d.png)

### Esri Land Cover Map

[Esri Land Cover Map](https://livingatlas.arcgis.com/landcover/), represents a prominent contribution to the field of geospatial analytics. Developed by Esri, a leading provider of Geographic Information System (GIS) technology, this map encompasses a diverse range of land cover classes derived from satellite imagery and advanced analytical techniques. 

- trained using billions of human-labeled image pixels from the National Geographic Society
- 9 land cover classes, description can be found [here](https://www.arcgis.com/home/item.html?id=cfcb7609de5f478eb7666240902d4d3d#overview)
- 10m resolution

![](https://gitlab.jsc.fz-juelich.de/hedgedoc/uploads/60e4e5bd-17b9-42e0-981b-763849ca5945.png)

- You need to download the tile using "wget" [Download link](https://livingatlas.arcgis.com/landcoverexplorer/#mapCenter=-95.819%2C29.689%2C11&mode=step&timeExtent=2017%2C2022&year=2022)

## Visual Interpretation

- Visual interpretation stands as a qualitative approach to validation. In this method, we can visually inspect and compare the generated land cover maps with high-resolution satellite imagery or available ground truth data. It allows for a direct assessment of the map's alignment with real-world features, providing valuable insights into its accuracy.

- Python script for visualization is here: /p/project/training2328/tian1/

- Use software like [QGIS](https://qgis.org/en/site/).

## Classification Accuracy Assessment

- In an accuracy assessment, the classification map is compared with validation data
- Based on the sampling protocol for the collected validation data, a **confusion matrix** can be constructed. It is a cross-tabulation of the class labels allocated by map and validation data
  - Usually the map classes are represented in rows and the reference classes in columns. 
- The confusion matrix allows for the calculation of the following accuracy metrics : 
   - Overall accuracy & Overall error  
   - Producer's accuracy 
   - User's accuracy 
   - Errors of omission 
   - Errors of commission 

- The table below gives an example of a confusion matrix for a classification with 3 classes (water, forest and urban). We will use this matrix to illustrate how to calculate the various accuracy metrics.


|Classified\Reference| Water | Forest | Urban | Total |
|:--:|:--:|:--:|:--:|:--:|
| Water|25|	5|	0|	**30**|
| Forest|7|	32|	2|	**41**|
| Urban|6|	3|	26|	**35**|
| Total|**38**|	**40**|	**28**|	**106**|

---

## Hands-on: Evaluating Your Trained Model

### Step 1: Load Your Model from Lab 5

```python
import torch
from lab4_1_transformer_training import TransformerModel

# Load checkpoint from Lab 5
checkpoint_path = "path_to_your_checkpoint/lab5_transformer_ddp.ckpt"
model = TransformerModel.load_from_checkpoint(checkpoint_path)
model.eval()  # Set to evaluation mode
```

### Step 2: Run Inference on Validation Set

```python
import numpy as np
from torch.utils.data import DataLoader

# Load validation data
val_loader = DataLoader(val_dataset, batch_size=512)

# Collect predictions
all_predictions = []
all_labels = []

with torch.no_grad():
    for batch in val_loader:
        x, y = batch
        x = x.view(x.size(0), -1, 10)
        logits = model(x, x)
        preds = torch.argmax(logits, dim=-1)
        
        all_predictions.extend(preds.cpu().numpy())
        all_labels.extend(y.cpu().numpy())

y_true = np.array(all_labels)
y_pred = np.array(all_predictions)
```

### Step 3: Calculate Confusion Matrix

```python
from sklearn.metrics import confusion_matrix, classification_report

# Compute confusion matrix
cm = confusion_matrix(y_true, y_pred)

# Print classification report
print(classification_report(y_true, y_pred, 
                          target_names=[f'Class {i}' for i in range(12)]))

# Visualize confusion matrix
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix - Your Transformer Model')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.show()
```

### Step 4: Save Predictions as GeoTIFF

```python
# Create prediction map
prediction_map = y_pred.reshape(10980, 10980)  # Reshape to original tile size

# Save as GeoTIFF with georeference from original S2 file
def save_predictions_as_geotiff(predictions, original_s2_path, output_path):
    from osgeo import gdal
    
    # Open reference S2 file for georeferencing
    source_ds = gdal.Open(original_s2_path)
    
    # Create output file
    driver = gdal.GetDriverByName('GTiff')
    out_ds = driver.Create(output_path, predictions.shape[1], 
                          predictions.shape[0], 1, gdal.GDT_Byte)
    
    # Copy geotransform and projection
    out_ds.SetGeoTransform(source_ds.GetGeoTransform())
    out_ds.SetProjection(source_ds.GetProjection())
    
    # Write predictions
    out_band = out_ds.GetRasterBand(1)
    out_band.WriteArray(predictions.astype(np.uint8))
    out_ds.FlushCache()
    
    print(f"Saved predictions to {output_path}")

save_predictions_as_geotiff(prediction_map, "original_s2.tif", 
                           "my_predictions.tif")
```

### Step 5: Compare with Other Land Cover Products

```python
# Load WorldCover and Esri data (align to same grid first)
worldcover = gdal.Open("worldcover_tile.tif").ReadAsArray()
esri = gdal.Open("esri_tile.tif").ReadAsArray()

# Align to your prediction (class mapping may differ)
# Create comparison plots
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].imshow(prediction_map)
axes[0].set_title('Your Transformer Model')

axes[1].imshow(worldcover)
axes[1].set_title('WorldCover 2021')

axes[2].imshow(esri)
axes[2].set_title('Esri Land Cover')

plt.show()
```

---

## Summary

This lab covered:

1. **Accuracy Assessment**: Confusion matrices for multi-class classification
2. **Accuracy Metrics**: OA, PA, UA, errors of omission and commission
3. **Land Cover Comparisons**: Benchmarking against WorldCover and Esri
4. **Visual Validation**: QGIS inspection of predictions
5. **Quantitative Reporting**: Detailed accuracy reports per class

### Your Classification Accuracy

After computing the confusion matrix from your Lab 5 model:

| Metric | Value |
|--------|-------|
| Overall Accuracy | [Your OA] |
| Kappa | [Your Kappa] |
| Weighted F1-Score | [Your F1] |

---

## Next Steps: Lab 7 - Foundation Models & TerraToRCH

### Why Foundation Models?

Your custom transformer (Labs 4-6) achieved ~XX% accuracy.  
Foundation models pre-trained on massive datasets can do better!

**Foundation Models for Remote Sensing:**
- **TerraToRCH**: Pre-trained vision transformer on Sentinel-2 data
- **Prithvi**: NASA's geospatial foundation model
- **MOSAIK**: Microsoft's multi-sensor pre-trained model

### What Lab 7 Covers

1. **Foundation Model Basics**
   - Pre-training vs. fine-tuning
   - Transfer learning for remote sensing

2. **TerraToRCH Integration**
   - Loading pre-trained TerraToRCH backbone
   - Custom classifier head for your data
   - Fine-tuning on Sentinel-2 patches

3. **Comparative Analysis**
   - Train custom transformer (Lab 4.1 approach)
   - Train with TerraToRCH backbone
   - Compare accuracy and training time
   - Discuss trade-offs

4. **Production Deployment**
   - Model quantization and optimization
   - Inference on large mosaics
   - Integration with geospatial pipelines

### Expected Improvements

| Model | OA | Training Time |
|-------|----|----|
| Custom Transformer | ~75-80% | 6 hours (1 GPU) |
| TerraToRCH Fine-tuned | ~85-90% | 2 hours (1 GPU) |
| Prithvi Fine-tuned | ~82-87% | 2 hours (1 GPU) |

---

## Prepare for Lab 7

Before starting Lab 7, complete:

1. ✅ Document your Lab 6 metrics
2. ✅ Save your best checkpoint
3. ✅ Create comparison visualizations
4. ✅ Write a brief accuracy assessment

### Pre-Lab Installation

```bash
# Install TerraToRCH and dependencies
pip install terratorch
pip install timm  # Required for vision transformers
pip install rasterio rio-cogeo  # Geospatial I/O
```

---

**Continue to Lab 7: Foundation Models & TerraToRCH →**

### Overall accuracy and overall error

- The **Overall accuracy (OA)** gives the proportion of validation samples correctly classified
- The **diagonal of the confusion matrix contains the correctly classified samples**. 
- To calculate OA sum up the number of correctly classified saples and divide it by the total number of reference samples 

$$
\text { Overall Accuracy }=\frac{(25+32+26)}{106}=0.78
$$

- The **Overall error** represents the proportion of validation samples that were classified incorrectly (i.e., complement of OA, OA+OE=100%)
- This is thus the complement of the overall accuracy (accuracy + error = 100%)
- So you can calculate OE from the OA, or you add the number of incorrectly classified samples and divide it by the total number of reference samples

$$
\text { Overall Error }=1-\text { Overall Accuracy }=1-0.78=0.22
$$

$$
\text { Overall Error }=\frac{(7+6+5+3+0+2)}{106}=0.22
$$

### Producer's accuracy (recall)

- The **Producer's accuracy (PA)** is a measure for how often real features on the ground are correctly shown on the classified map
  - It is the map accuracy from the point of view of the mapmaker (producer) 

- PA is also known as sensitivity (in statistics) and recall (in machine learning), 

- It is calculated per class by dividing the number of correctly classified reference samples by the total number of reference samples for that class (= column totals). 

$$
\text { PAwater }=\frac{25}{38}=0.66
$$

### User's accuracy (precision)

- The **User's accuracy (UA)** is a measure of how often the class on the map will actually be present on the ground.
  - So it is the map accuracy from the point of view of the map user.

- UA is also known (in statistics and machine learning) as precision 

- It is calculated per class by dividing the number of correct classifications by the total number of classified samples for that class (= row totals).

$$
U A \text { water }=\frac{25}{30}=0.83
$$

### Error of omission

- The **error of omission** is the proportion of reference samples that were left out (or omitted) from the correct class in the classified map
  - Sometimes referred to as a Type II error or false negative rate 
- It is complementary to the producer's accuracy, but can also be calculated for each class by dividing the incorrectly classified reference sites by the total number of reference sites for that class.

$$
\text { Omission error water }=\frac{7+6}{38}=0.34
$$

$$
1-P A \text { water }=1-0.66=0.34
$$

### Error of commission

- The **error of commission** is the proportion of classified sites that were assigned (or committed) to the incorrect class in the classified map
   - Sometimes referred to as a Type I error or false discovery rate
- The commission error is complementary to the user's accuracy, but can also be calculated for each class by dividing the incorrectly classified sites by the total number of classified samples for that class.

$$
\text { Commission error water }=\frac{5+0}{30}=0.17
$$

$$
1-U A \text { water }=1-0.83=0.17
$$

## Hands-on Python Script

- Located at /p/project/training2328/tian1/validation_final.ipynb